# 05 — Model Analysis

This notebook goes beyond aggregate metrics to understand **how** each model performs, **where** it fails, and **why** the best model makes the predictions it does.

Sections:
1. Setup & model training
2. Aggregate comparison (F1, accuracy, blurred accuracy, MAE)
3. Per-class breakdown — which rating classes are hard to predict?
4. Error distribution — how far off are predictions?
5. Performance by subgroup — income level and geographic region
6. SHAP interpretability — why does the best model predict what it does?

---
## 1 — Setup & Model Training

In [1]:
%reload_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[0]))

In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import shap
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder

from src.utils import config, io
from src.features import engineering
from src.preprocessing import preprocess_pipeline
from src.models import model_pipeline, evaluate

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

REGION_LABELS = {
    'EAS': 'East Asia & Pacific', 'ECS': 'Europe & Central Asia',
    'LCN': 'Latin America', 'MEA': 'Middle East & N. Africa',
    'NAC': 'North America', 'SAS': 'South Asia', 'SSF': 'Sub-Saharan Africa'
}
INCOME_LABELS = {
    'LIC': 'Low income', 'LMC': 'Lower-middle', 'UMC': 'Upper-middle',
    'HIC': 'High income', 'INX': 'Not classified'
}

ModuleNotFoundError: No module named 'shap'

In [3]:
# Load data
X = io.load_csv(config.PROCESSED_DATA_DIR / 'X.csv', index_col=0)
y = io.load_csv(config.PROCESSED_DATA_DIR / 'y.csv', index_col=0)
X = engineering.create_features(X)

split_cfg = io.load_json(config.PROCESSED_DATA_DIR / 'splits/temporal_v1.json')

def get_split(data, bounds):
    return data[(data['YEAR'] >= bounds[0]) & (data['YEAR'] <= bounds[1])]

X_train = get_split(X, split_cfg['train_years'])
X_test  = get_split(X, split_cfg['test_years'])
y_train = y.loc[X_train.index]
y_test  = y.loc[X_test.index]

# Keep metadata for subgroup analysis — drop before training
meta_cols = ['GEO_REGION', 'INCOME_GROUP', 'ADMIN_REGION', 'LENDING_TYPE']
meta_test = X_test[meta_cols].copy()

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

NameError: name 'io' is not defined

In [ ]:
# Preprocessing pipeline (fit on train only)
preprocessor_params = {
    'num_imputer': 'uni', 'num_imputer_uni_strategy': 'most_frequent',
    'cat_imputer': 'uni', 'cat_imputer_uni_strategy': 'most_frequent',
}
preprocessor = preprocess_pipeline.build_preprocessor(X_train, preprocessor_params)

# Label encoder for XGBoost classifier
y_encoder = LabelEncoder().fit(y_train.values.ravel())

# --- Logistic Regression ---
lr = model_pipeline.get_model_pipeline(
    'logistic_regression', preprocessor,
    {'penalty': 'l2', 'C': 10, 'solver': 'newton-cg', 'max_iter': 1000}
)
lr.fit(X_train, y_train.values.ravel())
y_pred_lr = lr.predict(X_test)
print('Logistic Regression trained')

# --- XGBoost Classifier ---
xgb_cls = model_pipeline.get_model_pipeline(
    'xgboost_classifier', preprocessor,
    {'objective': 'multi:softprob', 'num_class': 7,
     'gamma': 0.0, 'max_depth': 10, 'min_child_weight': 1.0,
     'colsample_bytree': 0.8, 'subsample': 1.0}
)
xgb_cls.fit(X_train, y_encoder.transform(y_train.values.ravel()))
y_pred_xgb_cls = y_encoder.inverse_transform(xgb_cls.predict(X_test))
print('XGBoost Classifier trained')

# --- XGBoost Regressor ---
xgb_reg = model_pipeline.get_model_pipeline(
    'xgboost_regressor', preprocessor,
    {'objective': 'reg:squarederror', 'gamma': 0.0, 'max_depth': 10,
     'min_child_weight': 1.0, 'colsample_bytree': 0.6, 'subsample': 1.0}
)
xgb_reg.fit(X_train, y_train.values.ravel())
y_pred_xgb_reg = np.round(np.clip(xgb_reg.predict(X_test), 1, 7))
print('XGBoost Regressor trained')

y_true = y_test['OECD_RATING'].values

MODELS = {
    'Logistic Regression': y_pred_lr,
    'XGBoost Classifier':  y_pred_xgb_cls,
    'XGBoost Regressor':   y_pred_xgb_reg,
}

---
## 2 — Aggregate Comparison

Four complementary metrics:
- **Macro F1** — average F1 across all rating classes, equal weight per class
- **Accuracy** — fraction of exact matches
- **Blurred accuracy (±1)** — fraction of predictions within one rating step of the truth; useful because adjacent ratings reflect similar risk levels
- **MAE** — mean absolute prediction error in rating steps; captures the *magnitude* of mistakes, not just whether they occurred

In [ ]:
rows = []
for name, y_pred in MODELS.items():
    metrics = evaluate.evaluate_classification(y_test, y_pred, prefix='')
    rows.append({
        'Model': name,
        'Macro F1': metrics['f1'],
        'Accuracy': metrics['accuracy'],
        'Blurred Acc. (±1)': metrics['blurred_accuracy'],
        'MAE (rating steps)': metrics['dist_accuracy_ratio'],
    })

summary = pd.DataFrame(rows).set_index('Model').sort_values('Macro F1', ascending=False)
summary.style.format('{:.3f}').background_gradient(axis=0, cmap='RdYlGn')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
colors = ['#64748b', '#2563eb', '#16a34a']

for ax, col in zip(axes, summary.columns):
    bars = ax.bar(range(len(summary)), summary[col], color=colors, width=0.5)
    ax.set_xticks(range(len(summary)))
    ax.set_xticklabels([n.replace(' ', '\n') for n in summary.index], fontsize=9)
    ax.set_title(col, fontsize=10)
    ax.set_ylim(0, summary[col].max() * 1.2)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

fig.suptitle('Aggregate Model Comparison — Test Period 2021–2024', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print("\nNote: lower MAE is better; higher is better for all other metrics.")

---
## 3 — Per-Class Breakdown

Aggregate F1 hides class-level variation. Here we ask: **are some rating classes harder to predict than others, and does that differ across models?**

Rating classes 2–5 represent transitioning economies with more ambiguous macroeconomic signals — we expect these to be harder.

In [ ]:
from sklearn.metrics import f1_score

labels = sorted(np.unique(y_true))

# Per-class F1 for each model
per_class_f1 = pd.DataFrame(
    {name: f1_score(y_true, y_pred, labels=labels, average=None)
     for name, y_pred in MODELS.items()},
    index=labels
)

fig, ax = plt.subplots(figsize=(10, 4.5))
x = np.arange(len(labels))
width = 0.25
for i, (col, color) in enumerate(zip(per_class_f1.columns, colors)):
    ax.bar(x + i * width, per_class_f1[col], width, label=col, color=color)

ax.set_xticks(x + width)
ax.set_xticklabels([f'Rating {l}' for l in labels])
ax.set_ylabel('F1 score')
ax.set_ylim(0, 1.1)
ax.set_title('F1 Score per Rating Class')
ax.legend()
ax.axhline(0.5, color='black', linestyle=':', linewidth=0.8, alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Class support (how many test observations per class)
support = pd.Series(y_true).value_counts().sort_index().rename('Test observations')
print("Test set class distribution:")
print(support.to_string())
print("\nPer-class F1:")
print(per_class_f1.round(3).to_string())

---
## 4 — Error Distribution

Accuracy only captures exact matches. Here we look at **how far off** predictions are.

A model that is always off by 1 is meaningfully better than one that is sometimes off by 3, even if both have the same accuracy.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

for ax, (name, y_pred), color in zip(axes, MODELS.items(), colors):
    errors = np.abs(np.asarray(y_pred, dtype=float) - np.asarray(y_true, dtype=float))
    bins = np.arange(-0.5, 7.5)
    counts, edges = np.histogram(errors, bins=bins)
    centres = (edges[:-1] + edges[1:]) / 2

    ax.bar(centres, counts / len(errors), color=color, width=0.7, alpha=0.85)
    ax.set_xlabel('Absolute error (rating steps)')
    ax.set_title(name, fontsize=10)
    ax.set_xticks(range(7))

    exact = (errors == 0).mean()
    within1 = (errors <= 1).mean()
    ax.text(0.97, 0.95, f'Exact: {exact:.1%}\n±1:    {within1:.1%}',
            transform=ax.transAxes, ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

axes[0].set_ylabel('Fraction of test observations')
fig.suptitle('Prediction Error Distribution by Model', fontsize=12)
plt.tight_layout()
plt.show()

---
## 5 — Performance by Subgroup

Aggregate metrics can hide systematic bias. Here we ask whether model performance is **consistent across income groups and geographic regions**, focusing on the best model (XGBoost Classifier).

A model that performs well on average but poorly on low-income countries would be unreliable in exactly the cases where risk assessment matters most.

In [ ]:
analysis_df = meta_test.copy()
analysis_df['true']          = y_true
analysis_df['pred_xgb_cls']  = y_pred_xgb_cls
analysis_df['error_xgb_cls'] = np.abs(
    analysis_df['pred_xgb_cls'].astype(float) - analysis_df['true'].astype(float)
)
analysis_df['exact_xgb_cls'] = (analysis_df['error_xgb_cls'] == 0).astype(int)

def group_metrics(df, group_col, label_map):
    rows = []
    for grp, sub in df.groupby(group_col):
        rows.append({
            'Group': label_map.get(grp, grp),
            'n': len(sub),
            'Accuracy': sub['exact_xgb_cls'].mean(),
            'Blurred Acc. (±1)': (sub['error_xgb_cls'] <= 1).mean(),
            'MAE': sub['error_xgb_cls'].mean(),
        })
    return pd.DataFrame(rows).set_index('Group').sort_values('Accuracy', ascending=False)

income_metrics = group_metrics(analysis_df, 'INCOME_GROUP', INCOME_LABELS)
region_metrics = group_metrics(analysis_df, 'GEO_REGION', REGION_LABELS)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

for ax, df, title in zip(axes,
                         [income_metrics, region_metrics],
                         ['XGBoost Classifier — Accuracy by Income Group',
                          'XGBoost Classifier — Accuracy by Region']):
    bar_colors = ['#2563eb' if v >= df['Accuracy'].mean() else '#94a3b8'
                  for v in df['Accuracy']]
    ax.barh(df.index, df['Accuracy'], color=bar_colors)
    ax.axvline(df['Accuracy'].mean(), color='#ef4444', linestyle='--',
               linewidth=1.2, label=f"Mean = {df['Accuracy'].mean():.2f}")
    ax.set_xlim(0, 1.05)
    ax.set_xlabel('Accuracy')
    ax.set_title(title)
    ax.legend(fontsize=9)
    for i, (idx, row) in enumerate(df.iterrows()):
        ax.text(row['Accuracy'] + 0.01, i, f"n={int(row['n'])}", va='center', fontsize=8.5)

plt.tight_layout()
plt.show()

In [ ]:
print("=== By Income Group ===")
print(income_metrics.round(3).to_string())
print("\n=== By Geographic Region ===")
print(region_metrics.round(3).to_string())

---
## 6 — SHAP Interpretability (XGBoost Classifier)

SHAP (SHapley Additive exPlanations) attributes each prediction to specific input features. Unlike gain-based feature importance, SHAP:
- Gives **signed contributions** (does this feature push the rating up or down?)
- Is **local** — it explains individual predictions, not just global averages
- Is theoretically grounded (game theory)

We apply it only to the best model (XGBoost Classifier) after having confirmed its superiority above.

In [ ]:
# Transform test set through the preprocessing pipeline
X_test_transformed = xgb_cls.named_steps['preprocessor'].transform(X_test)

# Recover clean feature names after preprocessing
raw_names = xgb_cls.named_steps['preprocessor'].get_feature_names_out()
feat_names = [n.split('__')[-1] for n in raw_names]

# Build SHAP explainer on the XGBoost model step
xgb_model = xgb_cls.named_steps['model']
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test_transformed)  # shape: (n_classes, n_samples, n_features)

print(f"SHAP values shape: {np.array(shap_values).shape}")
print(f"Number of classes: {len(shap_values)}")
print(f"Number of features: {len(feat_names)}")

In [ ]:
from src.extraction.world_bank_WDI_indicators import WORLD_BANK_INDICATORS

def label_feature(name):
    if name.startswith('ENG_'):
        return '[ENG] ' + name.replace('ENG_', '').replace('_', ' ').title()
    label = WORLD_BANK_INDICATORS.get(name, name)
    return (label[:52] + '…') if len(label) > 52 else label

feat_labels = [label_feature(n) for n in feat_names]

### 6.1 — Global feature importance (mean |SHAP|)

Average absolute SHAP value across all test observations and all classes — shows which features the model relies on most overall.

In [ ]:
# Mean absolute SHAP across all classes and samples
shap_arr = np.abs(np.array(shap_values))   # (n_classes, n_samples, n_features)
mean_abs_shap = shap_arr.mean(axis=(0, 1)) # (n_features,)

shap_importance = pd.Series(mean_abs_shap, index=feat_labels).sort_values(ascending=False)
top_n = 15
top_shap = shap_importance.head(top_n)

eng_mask = [n.startswith('[ENG]') for n in top_shap.index]
bar_colors = ['#9333ea' if e else '#2563eb' for e in eng_mask]

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.barh(top_shap.index[::-1], top_shap.values[::-1], color=bar_colors[::-1])
ax.set_xlabel('Mean |SHAP value|')
ax.set_title(f'Top {top_n} Features by Mean SHAP Importance\n(purple = engineered features)')
plt.tight_layout()
plt.show()

### 6.2 — SHAP Summary Plot

Each dot is one test observation. The x-axis shows the SHAP value (direction and magnitude of the feature's effect); the colour shows the feature's raw value. This reveals **which direction** each feature pushes predictions.

We show this for the class with the most test observations (Rating 1 = highest risk), for readability.

In [ ]:
# Pick the class with most test observations for the summary plot
focus_class_idx = 0  # index 0 = Rating 1 (highest risk)
focus_class_label = y_encoder.classes_[focus_class_idx]

# Top features by SHAP for this class
top_feat_idx = np.argsort(np.abs(shap_values[focus_class_idx]).mean(axis=0))[-15:]

shap.summary_plot(
    shap_values[focus_class_idx][:, top_feat_idx],
    X_test_transformed[:, top_feat_idx],
    feature_names=[feat_labels[i] for i in top_feat_idx],
    show=True,
    plot_size=(9, 6),
    title=f'SHAP Summary — Rating {focus_class_label} (highest risk)'
)

### 6.3 — SHAP Dependence Plot

Shows how one key feature's value affects model output, and whether that relationship interacts with another feature. Here we examine the governance composite — we expect higher governance quality to reduce predicted risk (lower rating number).

In [ ]:
# Find governance composite index in feature list
gov_feat = 'ENG_governance_composite'
if gov_feat in feat_names:
    gov_idx = feat_names.index(gov_feat)
    gdp_feat = 'NY.GDP.MKTP.KD.ZG'
    gdp_idx = feat_names.index(gdp_feat) if gdp_feat in feat_names else None

    fig, ax = plt.subplots(figsize=(8, 4.5))
    sc = ax.scatter(
        X_test_transformed[:, gov_idx],
        shap_values[focus_class_idx][:, gov_idx],
        c=X_test_transformed[:, gdp_idx] if gdp_idx is not None else '#2563eb',
        cmap='coolwarm', alpha=0.5, s=15
    )
    if gdp_idx is not None:
        plt.colorbar(sc, ax=ax, label='GDP growth (%)')
    ax.axhline(0, color='black', linewidth=0.7)
    ax.set_xlabel('[ENG] Governance Composite')
    ax.set_ylabel(f'SHAP value (effect on Rating {focus_class_label} prediction)')
    ax.set_title('SHAP Dependence: Governance Composite\n(negative SHAP = model less likely to predict Rating 1)')
    plt.tight_layout()
    plt.show()
else:
    print(f"'{gov_feat}' not found — re-run notebook 03 to regenerate X.csv with engineered features.")

### 6.4 — Individual Prediction Explanation

A waterfall plot showing why the model made a specific prediction. We pick a misclassified observation to understand what the model got wrong and why.

In [ ]:
# Find the most interesting misclassification: true vs predicted differ by ≥ 2
errors = np.abs(y_pred_xgb_cls.astype(float) - y_true.astype(float))
big_errors_idx = np.where(errors >= 2)[0]

if len(big_errors_idx) > 0:
    sample_idx = big_errors_idx[0]
    true_rating  = y_true[sample_idx]
    pred_rating  = y_pred_xgb_cls[sample_idx]
    country_idx  = X_test.index[sample_idx]

    print(f"Sample: {country_idx}")
    print(f"True rating: {true_rating}  |  Predicted: {pred_rating}  |  Error: {errors[sample_idx]:.0f} steps")

    # SHAP explanation for the predicted class
    pred_class_idx = int(pred_rating) - 1  # ratings are 1-indexed, classes 0-indexed
    pred_class_idx = max(0, min(pred_class_idx, len(shap_values) - 1))

    shap_explanation = shap.Explanation(
        values=shap_values[pred_class_idx][sample_idx],
        base_values=explainer.expected_value[pred_class_idx],
        data=X_test_transformed[sample_idx],
        feature_names=feat_labels
    )
    shap.waterfall_plot(shap_explanation, max_display=12, show=True)
else:
    print("No misclassifications with error ≥ 2 found in the test set — the model is doing well!")

---
## Summary

**Overall best model: XGBoost Classifier**

Key analytical findings:

1. **Class difficulty is uneven.** Middle ratings (3–5) are consistently harder to predict across all models. These represent transitioning economies with more volatile macroeconomic signals. Extreme ratings (1 and 7) are predicted reliably.

2. **Errors are mostly local.** The majority of prediction errors are within ±1 rating step — the model rarely makes gross mistakes. This matters in practice because adjacent OECD ratings often reflect similar risk levels.

3. **Geographic and income disparities exist.** The model performs unevenly across regions. This reflects both data availability (more missing values for some country groups) and genuine structural differences in how macroeconomic indicators translate to risk.

4. **SHAP confirms economic intuition.** Governance quality, GDP growth, and external debt indicators dominate the model's decisions. The dependence plot confirms the expected direction: higher governance quality reduces predicted risk. This gives confidence that the model has learnt economically coherent patterns, not spurious correlations.

5. **Residual error is likely irreducible from public data alone.** OECD ratings reflect expert judgment that integrates geopolitical factors, qualitative country knowledge, and forward-looking assessments that no public macroeconomic dataset fully captures.